# Bias Correction Review

Compares three evaporation correction approaches against the uncorrected baseline.
All four models are stepped forward independently from the same seed using the same
historical DB inputs — only the Q_evap calculation differs.

## Methods
| Method | What changes | Tune |
|---|---|---|
| `none` | Baseline — original C_E = 1.3e-3 | — |
| `evap_multiplier` | Scale C_E × multiplier | `CE_MULTIPLIER` (default 1.25) |
| `wind_floor` | Minimum effective wind speed | `EVAP_WIND_FLOOR_MS` (default 1.5 m/s) |
| `penman` | Penman 1948 combination equation | none |

## Decision criteria
- Which method brings the pool closest to ≤91°F on hot days without over-correcting on cool days?
- Which is most stable (monotonic response to temperature, no sign flips)?
- Which has the least additional bias introduced on mild days?

In [ ]:
import sqlite3, json, math, sys, os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

DB_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'wave_pool_weather.db'))
print(f'DB: {DB_PATH}  exists={os.path.exists(DB_PATH)}')

# Add scripts/ to path so we can import the thermal model directly
SCRIPTS_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts'))
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

REAL_MAX_F = 91.0
REAL_MAX_C = (REAL_MAX_F - 32) * 5 / 9

def c2f(c): return c * 9/5 + 32

In [ ]:
# ── Load historical inputs from DB ───────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

cur.execute("SELECT id FROM locations WHERE name='Waco'")
waco_id = cur.fetchone()['id']

cur.execute("""
    SELECT pt.date, pt.temp AS pool_C, pt.heat_fluxes,
           at.temp AS air_C, at.wind_speed, at.humidity, at.solar_radiation
    FROM pool_temps pt
    JOIN air_temps at ON pt.location_id=at.location_id AND pt.date=at.date
    WHERE pt.location_id=? AND pt.heat_fluxes IS NOT NULL
    ORDER BY pt.date
""", (waco_id,))
rows = cur.fetchall()
conn.close()
print(f'{len(rows)} rows  {rows[0]["date"]} → {rows[-1]["date"]}')

In [ ]:
# ── Inline thermal model — mirrors pull_api_data.py exactly ──────────────────
# Kept self-contained so the notebook runs without importing the full script.
SIGMA_SB      = 5.67e-8
EPSILON_WATER = 0.97
ALBEDO_WATER  = 0.06
RHO_WATER     = 1000.0
CP_WATER      = 4186.0
L_VAP         = 2.45e6
K_CONCRETE    = 1.4
L_CONCRETE_M  = 0.3048
K_SOIL_WACO   = 1.5
SOIL_DEPTH_M  = 2.0
DEPTH_M       = 2.0
GROUND_TEMP_C = 20.0

R_concrete = L_CONCRETE_M / K_CONCRETE
R_soil     = SOIL_DEPTH_M / K_SOIL_WACO
U_BOTTOM   = 1.0 / (R_concrete + R_soil)

def sat_vp(T_C):
    return 0.6108 * math.exp(17.27 * T_C / (T_C + 237.3))

def sky_eps(T_C, RH_pct):
    T_K   = T_C + 273.15
    RH    = max(0.0, min(1.0, RH_pct / 100.0))
    e_hPa = sat_vp(T_C) * RH * 10.0
    return min(1.0, 1.24 * (e_hPa / T_K) ** (1/7))

def penman_evap_Wm2(T_pool_C, T_air_C, RH_pct, wind_ms, solar_Wm2):
    """Penman (1948) open-water evaporation → W m⁻² (negative = cooling)."""
    RH      = max(0.0, min(1.0, RH_pct / 100.0))
    e_s_air = sat_vp(T_air_C)
    delta   = 4098.0 * e_s_air / (T_air_C + 237.3) ** 2
    gamma   = 0.067
    Rn      = solar_Wm2 * 86400.0 / 1e6  # MJ m⁻² day⁻¹
    lam     = 2.45
    f_u     = 6.43 * (1.0 + 0.536 * wind_ms)
    Ea      = f_u * (sat_vp(T_pool_C) - e_s_air * RH)
    E_mm    = (delta * (Rn / lam) + gamma * Ea) / (delta + gamma)
    return -(max(0.0, E_mm) / 1000.0 / 86400.0 * L_VAP)

def step(T_pool_C, T_air_C, RH_pct, wind_ms, solar_MJm2, method='none',
         ce_mult=1.25, wind_floor=1.5):
    T_pool_K = T_pool_C + 273.15
    T_air_K  = T_air_C  + 273.15
    RH       = max(0.0, min(1.0, RH_pct / 100.0))
    solar_Wm2 = max(0.0, float(solar_MJm2 or 15.0)) * 1e6 / 86400.0

    Q_solar  = (1.0 - ALBEDO_WATER) * solar_Wm2
    eps_sky  = sky_eps(T_air_C, RH_pct)
    Q_lw_net = eps_sky * SIGMA_SB * T_air_K**4 - EPSILON_WATER * SIGMA_SB * T_pool_K**4
    h_c      = 5.7 + 3.8 * wind_ms
    Q_conv   = h_c * (T_air_C - T_pool_C)

    EVAP_BASE = 1.2 * 1.3e-3 * L_VAP * 0.622 / 101.325
    e_s_pool  = sat_vp(T_pool_C)
    e_a       = sat_vp(T_air_C) * RH

    if method == 'evap_multiplier':
        Q_evap = -(EVAP_BASE * ce_mult) * wind_ms * (e_s_pool - e_a)
    elif method == 'wind_floor':
        Q_evap = -EVAP_BASE * max(wind_ms, wind_floor) * (e_s_pool - e_a)
    elif method == 'penman':
        Q_evap = penman_evap_Wm2(T_pool_C, T_air_C, RH_pct, wind_ms, solar_Wm2)
    else:
        Q_evap = -EVAP_BASE * wind_ms * (e_s_pool - e_a)

    Q_ground = -U_BOTTOM * (T_pool_C - GROUND_TEMP_C)
    Q_total  = Q_solar + Q_lw_net + Q_conv + Q_evap + Q_ground
    dT = Q_total * 86400.0 / (RHO_WATER * DEPTH_M * CP_WATER)
    return T_pool_C + dT, Q_evap, Q_solar

print('Model functions loaded.')

In [ ]:
# ── Run all four methods over the full historical record ──────────────────────
valid_rows = [r for r in rows if float(r['air_C']) < 45.0]
seed = float(valid_rows[0]['pool_C'])
T = {m: seed for m in ('none', 'evap_multiplier', 'wind_floor', 'penman')}

records = []
for r in valid_rows:
    air_C  = float(r['air_C'])
    RH     = float(r['humidity'] or 50.0)
    wind   = float(r['wind_speed'] or 3.0)
    solar  = float(r['solar_radiation']) if r['solar_radiation'] is not None else 15.0
    stored = float(r['pool_C'])

    result = {}
    for m in T:
        T[m], q_evap, q_solar = step(T[m], air_C, RH, wind, solar, method=m)
        result[m] = T[m]
    result['stored']  = stored
    result['air_C']   = air_C
    result['q_evap']  = q_evap
    result['q_solar'] = q_solar
    result['date']    = datetime.strptime(r['date'], '%Y-%m-%d')
    records.append(result)

dates       = np.array([r['date']                      for r in records])
stored_F    = np.array([c2f(r['stored'])               for r in records])
air_F       = np.array([c2f(r['air_C'])                for r in records])
none_F      = np.array([c2f(r['none'])                 for r in records])
mult_F      = np.array([c2f(r['evap_multiplier'])      for r in records])
floor_F     = np.array([c2f(r['wind_floor'])           for r in records])
penman_F    = np.array([c2f(r['penman'])               for r in records])

print(f'Baseline (none):          max={none_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(none_F>REAL_MAX_F).sum()}')
print(f'evap_multiplier (×1.25):  max={mult_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(mult_F>REAL_MAX_F).sum()}')
print(f'wind_floor (1.5 m/s):     max={floor_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(floor_F>REAL_MAX_F).sum()}')
print(f'penman:                   max={penman_F.max():.1f}°F  days>{REAL_MAX_F:.0f}°F: {(penman_F>REAL_MAX_F).sum()}')

## Plot 1 — Time series: all four methods vs. air temp

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(dates, air_F,    color='#e07b39', lw=1.2, alpha=0.6, label='Air temp')
ax.plot(dates, none_F,   color='#aaa',    lw=1.4, ls='--',   label='none (baseline)')
ax.plot(dates, mult_F,   color='#2196f3', lw=2.0,             label=f'evap_multiplier ×1.25')
ax.plot(dates, floor_F,  color='#43a047', lw=1.8, ls='-.',    label='wind_floor 1.5 m/s')
ax.plot(dates, penman_F, color='#ab47bc', lw=1.8, ls=':',     label='penman')
ax.axhline(REAL_MAX_F, color='red', lw=1.4, ls='--', label=f'Real-world cap {REAL_MAX_F}°F')

ax.set_ylabel('Pool temperature (°F)')
ax.set_title('Waco BSR — Bias Correction Method Comparison')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Plot 2 — Bias relative to cap (method - 91°F)
Positive = model exceeds real-world max. Zero is ideal on hot days.
Over-correction (large negative values) on cool days is also bad.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

pairs = [
    (axes[0], mult_F,   none_F, 'evap_multiplier ×1.25 vs baseline', '#2196f3'),
    (axes[1], floor_F,  none_F, 'wind_floor 1.5 m/s vs baseline',    '#43a047'),
    (axes[2], penman_F, none_F, 'penman vs baseline',                 '#ab47bc'),
]

for ax, method_F, base_F, title, color in pairs:
    delta = method_F - base_F
    ax.bar(dates, delta, color=np.where(delta < 0, color, '#ef5350'),
           alpha=0.8, width=0.8, label='cooling vs baseline (negative = more cooling)')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('ΔT (°F)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='y')

    bias_vs_cap = method_F - REAL_MAX_F
    hot = air_F > 90
    if hot.sum() > 0:
        ax.text(0.01, 0.97,
            f'Mean correction: {delta.mean():+.2f}°F  |  '
            f'Hot-day bias vs cap: {bias_vs_cap[hot].mean():+.1f}°F  |  '
            f'Days > 91°F: {(method_F > REAL_MAX_F).sum()}',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[2].xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Plot 3 — Pool temp vs. air temp scatter (all methods)
The ideal model tracks below the red line on hot days and stays
close to the 1:1 line on mild days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

methods = [
    (none_F,   'none (baseline)',          '#aaa'),
    (mult_F,   'evap_multiplier ×1.25',    '#2196f3'),
    (floor_F,  'wind_floor 1.5 m/s',       '#43a047'),
    (penman_F, 'penman',                    '#ab47bc'),
]

for ax, (pool, label, color) in zip(axes, methods):
    ax.scatter(air_F, pool, c=color, s=25, alpha=0.7, edgecolors='none')
    lim = [min(air_F.min(), pool.min()) - 2, max(air_F.max(), pool.max()) + 2]
    ax.plot(lim, lim, color='gray', ls=':', lw=1, label='1:1')
    ax.axhline(REAL_MAX_F, color='red', ls='--', lw=1.3, label=f'{REAL_MAX_F}°F cap')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Air temp (°F)')
    ax.set_ylabel('Pool temp (°F)')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    pct = 100 * (pool > REAL_MAX_F).mean()
    ax.text(0.02, 0.98, f'{pct:.0f}% days > cap', transform=ax.transAxes,
            va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))

plt.suptitle('Waco BSR — Pool vs Air Temp by Correction Method', fontsize=12)
plt.tight_layout()
plt.show()

## Plot 4 — Sensitivity: evap_multiplier CE value sweep
Shows steady-state pool temp at 105°F air as a function of the multiplier.
Use this to pick the CE_MULTIPLIER value before changing the default.

In [ ]:
def steady_state_F(method, ce_mult=1.25, wind_floor=1.5,
                   T_air_C=40.6, RH_pct=30.0, wind_ms=3.0, solar_MJm2=22.0):
    T = T_air_C - 2.0
    for _ in range(90):
        T, _, _ = step(T, T_air_C, RH_pct, wind_ms, solar_MJm2,
                       method=method, ce_mult=ce_mult, wind_floor=wind_floor)
    return c2f(T)

mults       = np.linspace(0.8, 2.5, 60)
wind_floors = np.linspace(0.5, 5.0, 60)

ss_mult  = [steady_state_F('evap_multiplier', ce_mult=m)    for m in mults]
ss_floor = [steady_state_F('wind_floor',      wind_floor=w) for w in wind_floors]
ss_penman = steady_state_F('penman')
ss_none   = steady_state_F('none')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Steady-State Pool Temp at T_air=105°F, RH=30%, Wind=3 m/s, Solar=22 MJ/m²/day')

ax1.plot(mults, ss_mult, color='#2196f3', lw=2)
ax1.axhline(REAL_MAX_F, color='red', ls='--', lw=1.3, label=f'{REAL_MAX_F}°F cap')
ax1.axhline(ss_none, color='gray', ls=':', lw=1.2, label=f'baseline={ss_none:.1f}°F')
ax1.axvline(1.25, color='black', ls='-.', lw=1, label='default mult=1.25')
crossings = np.where(np.diff(np.sign(np.array(ss_mult) - REAL_MAX_F)))[0]
for ci in crossings:
    ax1.axvline(mults[ci], color='green', ls='-', lw=1.5, label=f'crosses cap at ×{mults[ci]:.2f}')
ax1.fill_between(mults, ss_mult, REAL_MAX_F, where=np.array(ss_mult)>REAL_MAX_F,
                 color='red', alpha=0.12)
ax1.set_xlabel('CE_MULTIPLIER'); ax1.set_ylabel('Steady-state pool temp (°F)')
ax1.set_title('evap_multiplier sweep')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

ax2.plot(wind_floors, ss_floor, color='#43a047', lw=2)
ax2.axhline(REAL_MAX_F, color='red', ls='--', lw=1.3, label=f'{REAL_MAX_F}°F cap')
ax2.axhline(ss_none, color='gray', ls=':', lw=1.2, label=f'baseline={ss_none:.1f}°F')
ax2.axhline(ss_penman, color='#ab47bc', ls='--', lw=1.3, label=f'penman={ss_penman:.1f}°F')
ax2.axvline(1.5, color='black', ls='-.', lw=1, label='default floor=1.5 m/s')
crossings2 = np.where(np.diff(np.sign(np.array(ss_floor) - REAL_MAX_F)))[0]
for ci in crossings2:
    ax2.axvline(wind_floors[ci], color='green', ls='-', lw=1.5,
                label=f'crosses cap at {wind_floors[ci]:.1f} m/s')
ax2.fill_between(wind_floors, ss_floor, REAL_MAX_F, where=np.array(ss_floor)>REAL_MAX_F,
                 color='red', alpha=0.12)
ax2.set_xlabel('EVAP_WIND_FLOOR_MS (m/s)'); ax2.set_ylabel('Steady-state pool temp (°F)')
ax2.set_title('wind_floor sweep')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Penman steady-state at 105°F air: {ss_penman:.1f}°F')
print(f'Baseline steady-state:            {ss_none:.1f}°F')

## Summary table
Print key statistics for each method to support the go/no-go decision.

In [ ]:
hot = air_F > 90
mild = air_F < 75

print(f'{'Method':<22}  {'Max pool':>9}  {'Days>91F':>9}  {'HotBias':>9}  {'MildBias':>9}  {'MeanCorr':>9}')
print('-' * 75)
for label, pool in [('none (baseline)', none_F),
                    ('evap_multiplier ×1.25', mult_F),
                    ('wind_floor 1.5m/s', floor_F),
                    ('penman', penman_F)]:
    hot_bias  = (pool[hot]  - REAL_MAX_F).mean() if hot.sum()  else float('nan')
    mild_bias = (pool[mild] - none_F[mild]).mean() if mild.sum() else float('nan')
    corr = (pool - none_F).mean()
    print(f'{label:<22}  {pool.max():>8.1f}°F  {(pool>REAL_MAX_F).sum():>9}  '
          f'{hot_bias:>+8.1f}°F  {mild_bias:>+8.1f}°F  {corr:>+8.1f}°F')

print()
print('HotBias  = mean (pool - 91°F) on days where air > 90°F  (negative = under cap)')
print('MildBias = mean (method - baseline) on mild days         (should be near 0 or modest)')
print('MeanCorr = mean temperature correction across all days   (overall cooling applied)')